In [495]:
import pandas as pd
import numpy as np

In [496]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error

In [497]:
SEED: int = 42

# 2

Commonly, we instead split the data into a training set, a test set, and a validation set used to select hyperparameters. Use the provided dataset on white wine and divide it randomly into 70% train, 10% validation, and 20% test. The last column “quality” is the target, an integer from 1–10. We’ll view this as a regression problem with mean squared error as the desired loss function. Pick two model classes from the set {RandomForestRegressor, KNeighborsRegressor, MLPRegressor} and read their documentation in Scikit-learn to understand the hyperparameters of these models and what they do. Do grid search over a reasonably-large set of parameters (number of parameter sets >the number of points in the validation set): fit the model on the train set for each set of parameters, and select the “best parameters” using the score on the validation set. Finally, report the performance on the test set for each of the model classes you chose. What’s the relationship between the empirical risk on the validation set and the empirical risk on the test set?

In [498]:
fh_whitewine = 'data/winequality-white.csv'

df_whitewine = pd.read_csv(fh_whitewine, sep=';')

df_whitewine.head(3)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6


In [499]:
df_whitewine.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4898 entries, 0 to 4897
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         4898 non-null   float64
 1   volatile acidity      4898 non-null   float64
 2   citric acid           4898 non-null   float64
 3   residual sugar        4898 non-null   float64
 4   chlorides             4898 non-null   float64
 5   free sulfur dioxide   4898 non-null   float64
 6   total sulfur dioxide  4898 non-null   float64
 7   density               4898 non-null   float64
 8   pH                    4898 non-null   float64
 9   sulphates             4898 non-null   float64
 10  alcohol               4898 non-null   float64
 11  quality               4898 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 459.3 KB


In [500]:
df_whitewine = df_whitewine.drop_duplicates().reset_index(drop=True)

In [501]:
y = df_whitewine['quality'].values
X = df_whitewine.drop(columns='quality')

In [502]:
X_tr, X_temp, y_tr, y_temp = train_test_split(X, y, train_size=0.70, random_state=SEED)

X_val, X_te, y_val, y_te = train_test_split(X_temp, y_temp, test_size=2/3, random_state=SEED)

In [503]:
scaler = StandardScaler()

X_tr_s  = scaler.fit_transform(X_tr)
X_val_s = scaler.transform(X_val)
X_te_s  = scaler.transform(X_te)

In [504]:
print(f"Size of validation set: {len(X_val)}")

Size of validation set: 396


In [505]:
# Parameters for KNeighborsRegressor

knr_params = {
    'n_neighbors' : np.arange(1, 51, step = 1),       # Number of neighbors to use by default for kneighbors queries.
    'weights'     : ['uniform', 'distance'],          # Weight function used in prediction.
    'p'           : [1, 1.5, 2, 2.5, 3]               # Power parameter for the Minkowski metric. 
}

grid = list(ParameterGrid(knr_params))
len(grid)

500

In [506]:
# Grid search for KNeighborsRegressor

val_mse = []
for params in grid:
    knr = KNeighborsRegressor(n_jobs=-1, **params)
    knr.fit(X_tr_s, y_tr)
    predictions = knr.predict(X_val_s)

    val_mse.append(mean_squared_error(y_val, predictions))

optimal_param_idx = int(np.argmin(val_mse))
optimal_params = grid[optimal_param_idx]

optimal_params

{'n_neighbors': 6, 'p': 2, 'weights': 'distance'}

In [507]:
# Test MSE for KNeighborsRegressor

best_knr = KNeighborsRegressor(n_jobs=-1, **optimal_params)
best_knr.fit(X_tr_s, y_tr)

predictions = best_knr.predict(X_te_s)
test_mse = mean_squared_error(y_te, predictions)

predictions = best_knr.predict(X_val_s)
val_mse = mean_squared_error(y_val, predictions)

print(f"Test MSE: {test_mse:.4f}")
print(f"Validation MSE: {val_mse:.4f}")

Test MSE: 0.5535
Validation MSE: 0.5240


In [508]:
# Parameters for RandomForestRegressor

rfr_params = {
    'n_estimators'      : np.arange(5, 245, 25),     # The number of trees in the forest.
    'max_depth'         : [2, 4, 6, 8, 12, None],    # The maximum depth of the tree.
    'min_samples_split' : np.arange(2, 5),           # min_samples_split
    'max_features'      : np.arange(.1, 1, .2)       # The number of features to consider when looking for the best split.
}

grid = list(ParameterGrid(rfr_params))
len(grid)

900

In [509]:
# Grid search for RandomForestRegressor

val_mse = []
for params in grid:
    knr = RandomForestRegressor(n_jobs=-1, criterion='squared_error', **params, random_state=SEED)
    knr.fit(X_tr_s, y_tr)
    predictions = knr.predict(X_val_s)

    val_mse.append(mean_squared_error(y_val, predictions))

optimal_param_idx = int(np.argmin(val_mse))
optimal_params = grid[optimal_param_idx]

optimal_params

{'max_depth': None,
 'max_features': 0.1,
 'min_samples_split': 2,
 'n_estimators': 55}

In [510]:
# Test MSE for RandomForestRegressor

best_rfr = RandomForestRegressor(**optimal_params, n_jobs=-1, criterion='squared_error', random_state=SEED)
best_rfr.fit(X_tr_s, y_tr)

predictions = best_rfr.predict(X_te_s)
test_mse = mean_squared_error(y_te, predictions)

predictions = best_rfr.predict(X_val_s)
val_mse = mean_squared_error(y_val, predictions)

print(f"Test MSE: {test_mse:.4f}")
print(f"Validation MSE: {val_mse:.4f}")

Test MSE: 0.5162
Validation MSE: 0.5052


### Comment:



# 3

In class, we talked about the generalization behaviour of boosting as an increasing number of weak
classifiers are included in the model fitting.

(a) Use scikit learn to fit a gradient boosting model to dset0 (You should model the first column in dset0.csv given the other columns). Fit models with 1, 10, 100, 300, and 1000 trees. What do you observe about the generalization behaviour of these models?

(b) Use scikit learn to fit a gradient boosting model to dset1 (You should model the first column in
dset1.csv given the other columns). Fit models with 1, 10, 100, 300, and 1000 trees. What do you
observe about the generalization behaviour of these models?

(c) If your observations in parts (a) and (b) differ, explain what property of the data might explain
that.

### 3 (a)

In [511]:
n_trees = [1, 10, 100, 300, 1000]

In [512]:
fh_dset0 = 'data/dset0.csv'

dset0 = pd.read_csv(fh_dset0, header=None)

dset0.head(3)

,0,1,2,3,4,5
0,1.0,1.332,4.439,-0.487,-0.873,7.0
1,1.0,0.715,4.382,-2.142,3.567,-10.0
2,0.0,-1.545,3.921,-2.139,1.540,-2.0


In [513]:
y = dset0.iloc[:, 0].values
X = dset0.iloc[:, 1:]

In [514]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED)

In [515]:
dset0_GBC_accuracy = []
for trees in n_trees:
    model = GradientBoostingClassifier(n_estimators=trees, random_state=SEED)
    model.fit(X_tr, y_tr)

    tr_accuracy = np.mean(model.predict(X_tr) == y_tr)
    predictions = model.predict(X_te)
    te_accuracy = np.mean(predictions == y_te)

    dset0_GBC_accuracy.append((trees, tr_accuracy, te_accuracy))

pd.DataFrame(
    dset0_GBC_accuracy,
    columns = ['trees', 'train_accuracy', 'test_accuracy']
)

,trees,train_accuracy,test_accuracy
0,1,0.5750,0.57
1,10,0.8675,0.73
2,100,0.9950,0.82
3,300,1.0000,0.83
4,1000,1.0000,0.81


### Comment

Train accuracy increases significantly from 1 to 10 and from 10 to 100 trees, and we observe 100% train accuracy with 300 and 1000 trees.

Test accuracy improves significantly from 1 to 10 trees and improves moderately from 10 to 100 trees. However, when increasing the number of trees from 100 to 300, we only observe marginal improvements in test accuracy, and from 300 to 1000, test accuracy decreases.

Therefore, from our empirical results, we can conclude that 100 may be a reasonable number of estimators (trees).

### 3 (b)

In [516]:
n_trees = [1, 10, 100, 300, 1000]

In [517]:
fh_dset1 = 'data/dset1.csv'

dset1 = pd.read_csv(fh_dset1, header=None)

dset1.head(3)

,0,1,2,3,4,5
0,0.0,1.558,4.151,-2.741,-0.368,-4.0
1,1.0,-2.179,5.395,-2.018,-4.662,8.0
2,0.0,0.841,3.552,-1.664,3.957,-9.0


In [518]:
y = dset1.iloc[:, 0].values
X = dset1.iloc[:, 1:]

In [519]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED)

In [520]:
dset1_GBC_accuracy = []
for trees in n_trees:
    model = GradientBoostingClassifier(n_estimators=trees, random_state=SEED)
    model.fit(X_tr, y_tr)

    tr_accuracy = np.mean(model.predict(X_tr) == y_tr)
    predictions = model.predict(X_te)
    te_accuracy = np.mean(predictions == y_te)

    dset1_GBC_accuracy.append((trees, tr_accuracy, te_accuracy))

pd.DataFrame(
    dset1_GBC_accuracy,
    columns = ['trees', 'train_accuracy', 'test_accuracy']
)

,trees,train_accuracy,test_accuracy
0,1,0.5625,0.55
1,10,0.8725,0.77
2,100,0.9875,0.80
3,300,1.0000,0.82
4,1000,1.0000,0.82


### Comment

Similar to dset0, train accuracy increases significantly from 1 to 10 and from 10 to 100 trees, and we observe 100% train accuracy with 300 and 1000 trees.

Test accuracy improves significantly from 1 to 10 trees and improves moderately from 10 to 100 trees. However, when increasing the number of trees from 100 to 300, we only observe marginal improvements in test accuracy, and from 300 to 1000, test accuracy is constant.

Again, we can conclude that 100 may be a reasonable number of estimators (trees).

### 3 (c)

### Comment:

Both data sets showed similar results, and therefore I did not conduct any further data analysis to understand potential differences in the data. The minor variations in the results could be explained by the randomness of our samples.